# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "alumno01", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_emanuel_rb/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
### TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.

# Carga bronze_events
bronze_events = spark.table(qualified_table(config, "bronze_events"))

# Obtiene todas las columnas de bronze_events
bronze_columns = set(bronze_events.columns)

# Obtiene todos los tipos "event_type" de bronze_events
bronze_event_types = bronze_events.select("event_type").distinct()

# Carga challenge
challenge = spark.read.json(challenge_path)

# Obtiene todas las columnas de challenge
challenge_columns = set(challenge.columns)

# Obtiene todos los tipos "event_type" de bronze_events
challenge_event_types = challenge.select("event_type").distinct()

# Columnas nuevas dentro de Challenge
print("1. Campo nuevo en challenge: ", challenge_columns - bronze_columns)

# Nuevos valores de event_type
event_types = challenge_event_types.subtract(bronze_event_types).collect()
print("2. Tipos nuevos en challenge:", [fila.event_type for fila in event_types])

# Filas e identificadores distintos
print("3. Filas challenge:", challenge.count())
print("3. event_id distintos en challenge:", challenge.select("event_id").distinct().count())

1. Campo nuevo en challenge:  {'app_version'}
2. Tipos nuevos en challenge: ['refund']
3. Filas challenge: 10001
3. event_id distintos en challenge: 10000


## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

# TODO: unir por nombre admitiendo columnas faltantes, deduplicar y escribir target_table.
historico = spark.table(source_table)

nuevo = (
    spark.read.json(challenge_path)
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.lit(challenge_path))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    historico
    .unionByName(nuevo, allowMissingColumns=True)
    .dropDuplicates(["event_id"])
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(target_table, spark.table(target_table).count())

workspace.bigdata_emanuel_rb.bronze_events_v2 210000


## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.table(target_table)
total_rows = result.count()
distinct_ids = result.select("event_id").distinct().count()
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.

Detectar la evolución es darse cuenta de que el lote no coincide con la tabla. En este caso apareció la columna app_version, un event_type refund que no estaba, y un event_id duplicado. Eso se ve comparando esquemas y contando filas, antes de escribir nada.

Aceptarla técnicamente es poder incorporarla sin romper la historia. bronze_events_v2 une por nombre, deja app_version en null en los eventos viejos, se queda con un solo event_id y se puede volver a correr sin cambiar el total. Que el job termine bien no dice si el cambio es correcto.

Que sea válida para el negocio es otra decisión: si refund es un evento que la app realmente emite y si app_version sirve para analizar. Un campo nuevo puede cargarse bien y aun así ser un error del productor, o un cambio que nadie pidió.